In [3]:
import pandas as pd
import numpy as np
from datetime import datetime
import boto3
import io
import os
import warnings
import gc
warnings.filterwarnings("ignore")

# Chargement de la base 

In [6]:
def get_s3_client():
    """Client S3/MinIO"""
    return boto3.client(
        "s3",
        endpoint_url="http://minio.mon-namespace.svc.cluster.local:80",
        aws_access_key_id="wer1Or8j7hXa3QGk2M3M",
        aws_secret_access_key="YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt",
        verify=False
    )


def charger_panel_solde_1_2():
    """Charge les données depuis S3/MinIO"""
    bucket_name = "admindataanstat"
    object_key = "Solde/Panel_solde_complet_2015_2025.parquet"

    s3_client = get_s3_client()
    obj = s3_client.get_object(Bucket=bucket_name, Key=object_key)
    df = pd.read_parquet(io.BytesIO(obj["Body"].read()))
    
    print(f"✓ Données chargées : {len(df):,} observations")
    if "MATRICULE" in df.columns:
        print(f"✓ Matricules uniques : {df['MATRICULE'].nunique():,}")
    if "DATE_COLLECTE" in df.columns:
        print(f"✓ Période : {df['DATE_COLLECTE'].min()} → {df['DATE_COLLECTE'].max()}")
    
    return df


In [ ]:
panel = charger_panel_solde_1_2()


In [ ]:
panel.columns.tolist()


# CODE R 


In [1]:
panel = "C:/Users/f.migone/Desktop/projects/panel_admin_data/data/panel_solde_complet_2015_2025.parquet"

In [ ]:
# ==============================================================================
# CONSTRUCTION DES INDICATEURS DE SALAIRE - VERSION R OPTIMISÉE
# ==============================================================================
# Description: Calcul d'indicateurs salariaux par période, statut, sexe et catégorie
# Auteur: Romuald - CAE
# Date: Décembre 2024
# ==============================================================================

library(dplyr)
library(tidyr)
library(data.table)
library(writexl)
library(openxlsx)     # Alternative plus flexible pour Excel
library(furrr)        # Parallélisation
library(future)
library(tictoc)

# ==============================================================================
# DICTIONNAIRE DES MOIS
# ==============================================================================

MOIS_FR <- c(
  `1` = "Janvier", `2` = "Février", `3` = "Mars", `4` = "Avril",
  `5` = "Mai", `6` = "Juin", `7` = "Juillet", `8` = "Août",
  `9` = "Septembre", `10` = "Octobre", `11` = "Novembre", `12` = "Décembre"
)

# ==============================================================================
# FONCTION PRINCIPALE AVEC OPTIONS DE PERFORMANCE
# ==============================================================================

build_indicateurs_salaire <- function(
    panel,
    out_xlsx_path = "indicateurs_salaire_2015_2025.xlsx",
    out_csv_path = "indicateurs_salaire_2015_2025.csv",
    use_data_table = TRUE,      # TRUE = ultra-rapide pour gros volumes
    use_parallel = FALSE,        # TRUE = parallélisation des groupes
    n_cores = parallel::detectCores() - 1,
    use_openxlsx = FALSE,        # TRUE = Excel avec formatage
    verbose = TRUE
) {
  
  if (verbose) cat("🚀 Démarrage du calcul des indicateurs de salaire...\n")
  tic("Temps total")
  
  # --------------------------------------------------------------------------
  # 1. PRÉPARATION DES DONNÉES
  # --------------------------------------------------------------------------
  
  if (verbose) cat("📋 Préparation des données...\n")
  
  # Conversion en data.table si demandé (gain de performance significatif)
  if (use_data_table) {
    if (!inherits(panel, "data.table")) {
      df <- as.data.table(panel)
    } else {
      df <- copy(panel)
    }
  } else {
    df <- as.data.frame(panel)
  }
  
  # Détection automatique des colonnes
  ANNEE_COL <- if ("ANNEE_COLLECTE" %in% names(df)) "ANNEE_COLLECTE" else "YEAR"
  MOIS_COL <- if ("MOIS_COLLECTE" %in% names(df)) "MOIS_COLLECTE" else "MONTH"
  BRUT_COL <- if ("MONTANT BRUT_B1" %in% names(df)) "MONTANT BRUT_B1" else "MONTANT BRUT_B0"
  NET_COL <- "MONTANT NET"
  SEXE_COL <- "SEXE_IMPUTE"
  CAT_COL <- "CATEGORIE_1"
  MAT_COL <- "MATRICULE"
  
  # --------------------------------------------------------------------------
  # 2. NETTOYAGE ET TRANSFORMATION (VECTORISÉ)
  # --------------------------------------------------------------------------
  
  if (use_data_table) {
    # Version data.table (plus rapide)
    df[, `:=`(
      NET = as.numeric(get(NET_COL)),
      BRUT = as.numeric(get(BRUT_COL)),
      ANNEE = as.numeric(get(ANNEE_COL)),
      MOIS_NUM = as.numeric(get(MOIS_COL))
    )]
    
    # Mois en texte
    df[, MOIS := MOIS_FR[as.character(MOIS_NUM)]]
    
    # Sexe standardisé
    df[, SEXE_STD := fcase(
      toupper(get(SEXE_COL)) %in% c("M", "H", "HOMME"), "Homme",
      toupper(get(SEXE_COL)) %in% c("F", "FEMME"), "Femme",
      default = NA_character_
    )]
    
    # Statut
    df[, STATUT := fifelse(
      substr(as.character(get(MAT_COL)), 1, 1) %in% c("5", "6"),
      "Non fonctionnaire",
      "Fonctionnaire"
    )]
    
  } else {
    # Version dplyr (plus lisible)
    df <- df %>%
      mutate(
        NET = as.numeric(.data[[NET_COL]]),
        BRUT = as.numeric(.data[[BRUT_COL]]),
        ANNEE = as.numeric(.data[[ANNEE_COL]]),
        MOIS_NUM = as.numeric(.data[[MOIS_COL]]),
        MOIS = MOIS_FR[as.character(MOIS_NUM)],
        SEXE_STD = case_when(
          toupper(.data[[SEXE_COL]]) %in% c("M", "H", "HOMME") ~ "Homme",
          toupper(.data[[SEXE_COL]]) %in% c("F", "FEMME") ~ "Femme",
          TRUE ~ NA_character_
        ),
        STATUT = ifelse(
          substr(as.character(.data[[MAT_COL]]), 1, 1) %in% c("5", "6"),
          "Non fonctionnaire",
          "Fonctionnaire"
        )
      )
  }
  
  if (verbose) cat(sprintf("✅ Données préparées: %s lignes\n", nrow(df)))
  
  # --------------------------------------------------------------------------
  # 3. FONCTION D'AGRÉGATION
  # --------------------------------------------------------------------------
  
  calc_indicateurs <- function(data) {
    if (use_data_table && inherits(data, "data.table")) {
      # Version data.table ultra-rapide
      data[, .(
        `Salaire mensuel net moyen` = mean(NET, na.rm = TRUE),
        `Salaire mensuel brut moyen` = mean(BRUT, na.rm = TRUE),
        `Salaire médian mensuel net` = median(NET, na.rm = TRUE),
        `Salaire médian mensuel brut` = median(BRUT, na.rm = TRUE),
        `Min net` = min(NET, na.rm = TRUE),
        `Max net` = max(NET, na.rm = TRUE),
        `Effectif` = .N
      ), by = .(ANNEE, MOIS_NUM, MOIS)]
    } else {
      # Version dplyr
      data %>%
        group_by(ANNEE, MOIS_NUM, MOIS) %>%
        summarise(
          `Salaire mensuel net moyen` = mean(NET, na.rm = TRUE),
          `Salaire mensuel brut moyen` = mean(BRUT, na.rm = TRUE),
          `Salaire médian mensuel net` = median(NET, na.rm = TRUE),
          `Salaire médian mensuel brut` = median(BRUT, na.rm = TRUE),
          `Min net` = min(NET, na.rm = TRUE),
          `Max net` = max(NET, na.rm = TRUE),
          `Effectif` = n(),
          .groups = "drop"
        )
    }
  }
  
  # --------------------------------------------------------------------------
  # 4. CALCUL DES INDICATEURS PAR GROUPE
  # --------------------------------------------------------------------------
  
  if (verbose) cat("📊 Calcul des indicateurs par groupe...\n")
  
  frames_list <- list()
  
  # Configuration parallélisation si demandée
  if (use_parallel) {
    plan(multisession, workers = n_cores)
    if (verbose) cat(sprintf("⚡ Parallélisation activée avec %d cœurs\n", n_cores))
  }
  
  # A. Fonctionnaires par catégorie
  categories <- sort(unique(df[[CAT_COL]][!is.na(df[[CAT_COL]])]))
  
  if (use_data_table) {
    for (cat in categories) {
      nom_groupe <- sprintf("Fonctionnaire - Cat %s", cat)
      if (verbose) cat(sprintf("  → %s\n", nom_groupe))
      
      sous_df <- df[STATUT == "Fonctionnaire" & get(CAT_COL) == cat]
      frames_list[[nom_groupe]] <- calc_indicateurs(sous_df)
    }
  } else {
    if (use_parallel) {
      # Version parallèle
      frames_list_cat <- future_map(categories, function(cat) {
        nom_groupe <- sprintf("Fonctionnaire - Cat %s", cat)
        sous_df <- df %>% filter(STATUT == "Fonctionnaire", .data[[CAT_COL]] == cat)
        result <- calc_indicateurs(sous_df)
        list(nom = nom_groupe, data = result)
      })
      for (item in frames_list_cat) {
        frames_list[[item$nom]] <- item$data
      }
    } else {
      # Version séquentielle
      for (cat in categories) {
        nom_groupe <- sprintf("Fonctionnaire - Cat %s", cat)
        if (verbose) cat(sprintf("  → %s\n", nom_groupe))
        
        sous_df <- df %>% filter(STATUT == "Fonctionnaire", .data[[CAT_COL]] == cat)
        frames_list[[nom_groupe]] <- calc_indicateurs(sous_df)
      }
    }
  }
  
  # B. Non fonctionnaires
  if (verbose) cat("  → Non fonctionnaire\n")
  if (use_data_table) {
    frames_list[["Non fonctionnaire"]] <- calc_indicateurs(df[STATUT == "Non fonctionnaire"])
  } else {
    frames_list[["Non fonctionnaire"]] <- calc_indicateurs(df %>% filter(STATUT == "Non fonctionnaire"))
  }
  
  # C. Ensemble
  if (verbose) cat("  → Ensemble\n")
  frames_list[["Ensemble"]] <- calc_indicateurs(df)
  
  # D. Par sexe
  if (verbose) cat("  → Homme / Femme\n")
  if (use_data_table) {
    frames_list[["Homme"]] <- calc_indicateurs(df[SEXE_STD == "Homme"])
    frames_list[["Femme"]] <- calc_indicateurs(df[SEXE_STD == "Femme"])
  } else {
    frames_list[["Homme"]] <- calc_indicateurs(df %>% filter(SEXE_STD == "Homme"))
    frames_list[["Femme"]] <- calc_indicateurs(df %>% filter(SEXE_STD == "Femme"))
  }
  
  # Arrêt parallélisation
  if (use_parallel) {
    plan(sequential)
  }
  
  # --------------------------------------------------------------------------
  # 5. ASSEMBLAGE FINAL
  # --------------------------------------------------------------------------
  
  if (verbose) cat("🔧 Assemblage du tableau final...\n")
  
  # Combiner tous les résultats
  result_list <- lapply(names(frames_list), function(nom) {
    df_temp <- frames_list[[nom]]
    if (use_data_table && inherits(df_temp, "data.table")) {
      df_temp <- as.data.frame(df_temp)
    }
    
    # Renommer les colonnes avec préfixe
    cols_to_rename <- setdiff(names(df_temp), c("ANNEE", "MOIS_NUM", "MOIS"))
    names(df_temp)[names(df_temp) %in% cols_to_rename] <- 
      paste(nom, cols_to_rename, sep = " - ")
    
    df_temp
  })
  
  # Fusion par ANNEE, MOIS_NUM, MOIS
  result <- Reduce(function(x, y) {
    merge(x, y, by = c("ANNEE", "MOIS_NUM", "MOIS"), all = TRUE)
  }, result_list)
  
  # Tri final
  result <- result[order(result$ANNEE, result$MOIS_NUM), ]
  result <- result[, !names(result) %in% "MOIS_NUM"]
  
  if (verbose) cat(sprintf("✅ Tableau final: %s lignes × %s colonnes\n", 
                           nrow(result), ncol(result)))
  
  # --------------------------------------------------------------------------
  # 6. EXPORTS
  # --------------------------------------------------------------------------
  
  if (verbose) cat("💾 Export des fichiers...\n")
  
  # CSV (rapide)
  fwrite(result, out_csv_path, bom = TRUE)
  if (verbose) cat(sprintf("  ✓ CSV: %s\n", out_csv_path))
  
  # Excel avec options
  if (use_openxlsx) {
    # Version avec formatage (plus lent mais plus joli)
    wb <- createWorkbook()
    addWorksheet(wb, "Indicateurs")
    writeData(wb, "Indicateurs", result)
    
    # Style d'en-tête
    headerStyle <- createStyle(
      fontSize = 11,
      fontColour = "#FFFFFF",
      halign = "center",
      fgFill = "#4F81BD",
      border = "TopBottom",
      textDecoration = "bold"
    )
    addStyle(wb, "Indicateurs", headerStyle, rows = 1, cols = 1:ncol(result), gridExpand = TRUE)
    
    # Format nombres
    numStyle <- createStyle(numFmt = "#,##0")
    addStyle(wb, "Indicateurs", numStyle, rows = 2:(nrow(result) + 1), 
             cols = 3:ncol(result), gridExpand = TRUE)
    
    # Largeur colonnes
    setColWidths(wb, "Indicateurs", cols = 1:ncol(result), widths = "auto")
    
    saveWorkbook(wb, out_xlsx_path, overwrite = TRUE)
  } else {
    # Version rapide sans formatage
    write_xlsx(result, out_xlsx_path)
  }
  
  if (verbose) cat(sprintf("  ✓ Excel: %s\n", out_xlsx_path))
  
  toc()
  
  if (verbose) cat("✨ Terminé!\n")
  
  return(result)
}

# ==============================================================================
# FONCTION WRAPPER AVEC GESTION MÉMOIRE POUR TRÈS GROS VOLUMES
# ==============================================================================

build_indicateurs_salaire_big <- function(
    panel_path,  # Chemin vers fichier Parquet
    out_xlsx_path = "indicateurs_salaire_2015_2025.xlsx",
    out_csv_path = "indicateurs_salaire_2015_2025.csv",
    chunk_size = 1000000,
    verbose = TRUE
) {
  
  library(arrow)
  
  if (verbose) cat("📦 Chargement des données par chunks depuis Parquet...\n")
  
  # Ouvrir le dataset
  ds <- open_dataset(panel_path)
  
  # Calculer les indicateurs directement avec Arrow (sans tout charger en mémoire)
  result <- ds %>%
    # Transformations Arrow (lazy evaluation)
    mutate(
      ANNEE = as.numeric(ANNEE_COLLECTE),
      MOIS_NUM = as.numeric(MOIS_COLLECTE),
      NET = as.numeric(`MONTANT NET`),
      BRUT = as.numeric(`MONTANT BRUT_B1`)
    ) %>%
    group_by(ANNEE, MOIS_NUM) %>%
    summarise(
      `Salaire mensuel net moyen` = mean(NET, na.rm = TRUE),
      `Salaire mensuel brut moyen` = mean(BRUT, na.rm = TRUE),
      .groups = "drop"
    ) %>%
    collect()  # Charger seulement le résultat agrégé
  
  # Ajouter les mois
  result$MOIS <- MOIS_FR[as.character(result$MOIS_NUM)]
  
  # Exports
  fwrite(result, out_csv_path, bom = TRUE)
  write_xlsx(result, out_xlsx_path)
  
  return(result)
}

# ==============================================================================
# EXEMPLE D'UTILISATION
# ==============================================================================

if (FALSE) {
  
  # Cas 1: Données déjà en mémoire (DataFrame ou data.table)
  indicateurs <- build_indicateurs_salaire(
    panel = mon_panel,
    out_xlsx_path = "resultats/indicateurs_2015_2025.xlsx",
    out_csv_path = "resultats/indicateurs_2015_2025.csv",
    use_data_table = TRUE,      # Recommandé pour >1M lignes
    use_parallel = TRUE,         # Si >10M lignes
    n_cores = 6,
    use_openxlsx = TRUE,        # Pour joli formatage Excel
    verbose = TRUE
  )
  
  # Cas 2: Très gros fichier Parquet (>5M lignes)
  indicateurs <- build_indicateurs_salaire_big(
    panel_path = "data/panel_complet.parquet",
    out_xlsx_path = "indicateurs_big.xlsx",
    verbose = TRUE
  )
  
}

# Calcul des indicateurs 

In [ ]:
import pandas as pd
import numpy as np

MOIS_FR = {
    1: "Janvier", 2: "Février", 3: "Mars", 4: "Avril", 5: "Mai", 6: "Juin",
    7: "Juillet", 8: "Août", 9: "Septembre", 10: "Octobre", 11: "Novembre", 12: "Décembre"
}

def build_indicateurs_salaire(
    panel,
    out_xlsx_path="indicateurs_salaire_2015_2025.xlsx",
    out_csv_path="indicateurs_salaire_2015_2025.csv"
):
    df = panel.copy()

    # Colonnes (figées selon ta demande)
    ANNEE_COL = "ANNEE_COLLECTE" if "ANNEE_COLLECTE" in df.columns else "YEAR"
    MOIS_COL = "MOIS_COLLECTE" if "MOIS_COLLECTE" in df.columns else "MONTH"
    BRUT_COL = "MONTANT BRUT_B1" if "MONTANT BRUT_B1" in df.columns else "MONTANT BRUT_B0"
    NET_COL = "MONTANT NET"
    SEXE_COL = "SEXE_IMPUTE"
    CAT_COL = "CATEGORIE_1"
    MAT_COL = "MATRICULE"

    # Sécurisation types
    df[NET_COL] = pd.to_numeric(df[NET_COL], errors="coerce")
    df[BRUT_COL] = pd.to_numeric(df[BRUT_COL], errors="coerce")

    # Année / mois
    df["ANNEE"] = pd.to_numeric(df[ANNEE_COL], errors="coerce")
    df["MOIS_NUM"] = pd.to_numeric(df[MOIS_COL], errors="coerce")
    df["MOIS"] = df["MOIS_NUM"].map(MOIS_FR)

    # Sexe
    df["SEXE_STD"] = df[SEXE_COL].astype(str).str.upper().map({
        "M": "Homme", "H": "Homme", "HOMME": "Homme",
        "F": "Femme", "FEMME": "Femme"
    })

    # Statut
    df["STATUT"] = np.where(
        df[MAT_COL].astype(str).str.startswith(("5", "6")),
        "Non fonctionnaire",
        "Fonctionnaire"
    )

    # Fonction d’agrégation
    def agg(g):
        return pd.Series({
            "Salaire mensuel net moyen": g[NET_COL].mean(),
            "Salaire mensuel brut moyen": g[BRUT_COL].mean(),
            "Salaire médian mensuel net": g[NET_COL].median(),
            "Salaire médian mensuel brut": g[BRUT_COL].median(),
            "Min net": g[NET_COL].min(),
            "Max net": g[NET_COL].max(),
        })

    key = ["ANNEE", "MOIS_NUM", "MOIS"]
    frames = {}

    # Fonctionnaires par catégorie
    for cat in sorted(df[CAT_COL].dropna().unique()):
        frames[f"Fonctionnaire - Cat {cat}"] = (
            df[(df["STATUT"] == "Fonctionnaire") & (df[CAT_COL] == cat)]
            .groupby(key)
            .apply(agg)
        )

    # Non fonctionnaires
    frames["Non fonctionnaire"] = (
        df[df["STATUT"] == "Non fonctionnaire"]
        .groupby(key)
        .apply(agg)
    )

    # Ensemble
    frames["Ensemble"] = df.groupby(key).apply(agg)

    # Sexe
    frames["Homme"] = df[df["SEXE_STD"] == "Homme"].groupby(key).apply(agg)
    frames["Femme"] = df[df["SEXE_STD"] == "Femme"].groupby(key).apply(agg)

    # Tableau final
    result = pd.concat(frames, axis=1).reset_index()
    result = result.sort_values(["ANNEE", "MOIS_NUM"]).drop(columns="MOIS_NUM")

    # Exports
    result.to_excel(out_xlsx_path, index=False)
    result.to_csv(out_csv_path, index=False, encoding="utf-8-sig")

    return result


In [ ]:
# Charger les données
panel = charger_panel_solde_1_2()

# Lancer l'indicateur
table_indicateurs = build_indicateurs_salaire(
    panel,
    out_xlsx_path="indicateurs_salaire_2015_2025.xlsx",
    out_csv_path="indicateurs_salaire_2015_2025.csv"
)

table_indicateurs.head()
    